In [1]:
import logging
import os

logger = logging.getLogger(__name__)

def parse(line):
    vectorStart = line.rfind('[')
    vectorEnd = line.rfind(']')
    text = line[0:vectorStart].strip()
    vectorText = line[vectorStart+2:vectorEnd]
    
    vector = []
    for item in vectorText.split():
        vector.append(float(item))
    return (text, vector)

class KnowledgeBase:

    def __init__(self, filename):
        self.filename = filename
        self.file = None
        self.nextLine = None
        self.currentLine = None
        self.lineCounter = 0

    def __iter__(self):
        return self

    def __next__(self):
        if not self.file:
            logger.info(f"opening knowledgebase {self.filename} for reading.")
            self.file = open(self.filename, "r")
            self.nextLine = self.file.readline()
            self.currentLine = None
            self.lineCounter = 0
        if not self.nextLine:
            logger.info(f"no more facts in {self.filename}.")
            self.file.close()
            self.file = None
            raise StopIteration
        self.currentLine = parse(self.nextLine)
        self.nextLine = self.file.readline()
        self.lineCounter += 1
        logger.debug(f"reading fact {self.lineCounter} from {self.filename}.")
        return self.currentLine

    def load(self):
        pass
        
    def reset(self):
        try:
            os.remove(self.filename)
            logger.warning(f"removed knowledge base {self.filename}")
        except OSError:
            pass
            
    def insert(self, text, encoding):
        if not self.file:
            logger.info(f"opening knowledge base {self.filename}.")
            self.file = open(self.filename, "a")
        logger.debug(f"writing fact \"{text}\" to {self.filename}.")
        # this should go into an encode def to match the parse def.
        self.file.write(text.rstrip())
        self.file.write(" [ ")
        for value in encoding:
            self.file.write(f"{value:.8f} ")
        self.file.write("]\n")
        self.file.flush()

    def close(self):
        logger.info(f"closing knowledge base {self.filename}.")
        self.file.close()
        self.file = None